In [3]:
import json
from datetime import date
from pathlib import Path

import numpy as np
import polars as pl
import yaml

from common import read_day, daterange


def clean_os_version(x) -> float:
    result = x
    if "." in x:
        result = x.strip().split(".")[0]
    if x.strip() in ["ANDROID", "IOS", "Other", "null", ""]:
        result = np.nan
    return float(result)


with open("../config.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [2]:
with open("../output/id_lookup/id_lookup.json", "r") as fp:
    lookup = json.load(fp)

with open("../data/os_version.yaml", "r") as fp:
    os_versions = yaml.safe_load(fp)
os_versions["ios"] = {k: v.strftime("%Y-%m-%d") for k, v in os_versions["ios"].items()}
os_versions["android"] = {
    k: v.strftime("%Y-%m-%d") if isinstance(v, date) else v
    for k, v in os_versions["android"].items()
}

In [4]:
days = (
    list(daterange(date(2023, 10, 1), date(2023, 12, 31)))
    + list(daterange(date(2024, 1, 1), date(2024, 2, 29)))
    + list(daterange(date(2024, 10, 1), date(2024, 12, 31)))
    + list(daterange(date(2025, 10, 1), date(2025, 10, 31)))
)

In [13]:
to_drop = [
    # "aid_type",
    "horizontal_accuracy",
    "altitude",
    "altitude_accuracy",
    "ip",
    # "user_agent",
    "model",
    "carrier",
    "OS",
    "OS_version",
    "manufacturer",
    "location_method",
]

In [10]:
OUTPUT = "/mnt/common-hdd/group-dirs/tkom-2026-project/cleaned/"

In [17]:
# for i in days:
for i in daterange(date(2023, 12, 1), date(2023, 12, 31)):
    data = read_day(i, config["dirs"]["pickwell"])
    data = data.with_columns(pl.from_epoch(pl.col("timestamp"), time_unit="s"))

    # Clean OS version
    if "OS_version" not in to_drop:
        data = data.with_columns(
            pl.col("OS_version")
            .fill_null("")
            .map_elements(clean_os_version, return_dtype=float)
        )

    # Drop insignificant columns
    data = data.drop(to_drop)

    # Set timezone
    data = data.with_columns(
        timestamp=pl.col("timestamp").dt.convert_time_zone(time_zone="Europe/Budapest"),
    )

    # Replace device IDs
    data = data.with_columns(pl.col("device_aid_hashed").replace_strict(lookup))
    data = data.rename({"device_aid_hashed": "device_id"})
    path = f"{OUTPUT}/{i.strftime('%Y/%m/%d')}"
    Path(path).mkdir(exist_ok=True, parents=True)
    data.sort("timestamp").write_csv(
        f"{path}/locations.csv", datetime_format="%Y-%m-%d %H:%M:%S %z"
    )